# Import dependencies

In [1]:
import os, sys

sys.path.append(os.path.abspath("../PDFair"))

import PDFair
import eval
import visualize
import types
import glob
import re
import pandas as pd
import os


[0715 16:41.19 @utils.py:161]  INF  NumExpr defaulting to 16 threads.
/home/marty/miniconda3/envs/test/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[0715 16:41.22 @dd.py:422]  INF  Config: 
 {'DEVICE': 'cpu',
 'LANGUAGE': 'nld',
 'LAYOUT_NMS_PAIRS': {'COMBINATIONS': None, 'PRIORITY': None, 'THRESHOLDS': None},
 'LIB': 'PT',
 'OCR': {'CONFIG': {'TESSERACT': 'dd/conf_tesseract.yaml'},
         'USE_DOCTR': False,
         'USE_TESSERACT': True,
         'USE_TEXTRACT': False,
         'WEIGHTS': {'DOCTR_RECOGNITION': {'PT': 'doctr/crnn_vgg16_bn/pt/crnn_vgg16_bn-9762b0b0.pt',
                                           'TF': 'doctr/crnn_vgg16_bn/tf/crnn_vgg16_bn-76b7f2c6.zip'},
                     'DOCTR_WORD': {'PT': 'doctr/db_resnet50/pt/db_resnet50-ac60cadc.pt',
                                    'TF

# Running PDFair (in batches)
## Loading dataframe

In [2]:
df = pd.DataFrame({
        'dc_source': glob.glob('Pdfs/*'),
        'md': None
    })

df

,dc_source,md
0,Pdfs/nl.mnre1010.2i.2021.24.misc.1.pdf,None
1,Pdfs/nl.mnre1045.2i.2020.3.misc.1.pdf,None
2,Pdfs/nl.mnre1045.2i.2020.76.misc.1.pdf,None
3,Pdfs/nl.mnre1045.2i.2021.16.misc.1.pdf,None
4,Pdfs/nl.mnre1045.2i.2022.32.misc.1.pdf,None


## Postprocessing functions

In [3]:
# MD: 'A' tag creation on link
pattern_url = r'\b((http(s)?:\/\/)?(www\.)?[-a-zA-Z0-9@:%._\+~#=]{2,256}\.[a-z]{2,6}\b([-a-zA-Z0-9@:%_\+.~#?&//=]*))\b'

def md_url(match):
    url = match.group(1)
    return f'[{url}]({url})'

# MD: remove double list when not necessary ("* -" > "* ")
pattern_double_list = r'\n((\*|-|\+) (\*|-|\+)) '

def md_double_list(match):
    # url = match.group(1)
    return f'\n* '

def postprocess(text):
    return re.sub(pattern_double_list, md_double_list, re.sub(pattern_url, md_url, text))


## Processing the PDFs

In [14]:
offset = 0
run_n_times = 2
batchsize = 3

def getMd(path):
        try:
            pdf = PDFair.Pdf(path)  # Create Pdf class
            pdf.pdf2doc(max=3)  # Run deepdoctection on first 3 pages (if available)
            for i, _ in enumerate(pdf.pages):
                pdf.pages[i].detect_header()  # Detect headers for page
                pdf.pages[i].doc2md(skip_headers=True)  # Generate markdown (exclude headers)

            return [postprocess(page.md) for page in pdf.pages]  # Postprocess markdown
        except Exception as e:
            return ['FAILED:' + str(e)]

def batch(df, offset, size):
    apply = df[offset:offset+size]  # Create df of batch
    apply['md'] = apply.apply(lambda x: getMd(x['dc_source']), axis=1)  # Generate markdown
    apply.to_json(f'results[{offset}-{offset+len(apply)-1}].json', orient='records', indent=2)  # Save batch

for i in range(offset, offset + batchsize * run_n_times, batchsize):
    batch(df, i, batchsize)

[0715 14:06.03 @doctectionpipe.py:84]  INF  Processing nl.mnre1010.2i.2021.24.misc.1_0.pdf
/home/marty/miniconda3/envs/deepdoc2/lib/python3.9/site-packages/torch/nn/modules/module.py:1527: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3526.)
  return forward_call(*args, **kwargs)
[0715 14:06.09 @context.py:126]  INF  ImageLayoutService total: 4.5387 sec.
[0715 14:06.09 @context.py:126]  INF  SubImageLayoutService total: 0.0001 sec.
[0715 14:06.09 @context.py:126]  INF  SubImageLayoutService total: 0.0 sec.
[0715 14:06.09 @context.py:126]  INF  TableSegmentationService total: 0.0001 sec.
[0715 14:06.09 @context.py:126]  INF  TableSegmentationRefinementService total: 0.0 sec.
[0715 14:06.09 @context.py:126]  INF  TextExtractionService total: 0.1837 sec.
[0715 14:06.09 @context.py:126]  INF  TextExtractionService total: 0.0003 sec.
[0715 14:06.09 @context.py:126]  INF

## Merging results

In [15]:
results = pd.concat(pd.read_json(file) for file in glob.glob('results*.json')).reset_index(drop=True)

results

,dc_source,md
0,Pdfs/nl.mnre1010.2i.2021.24.misc.1.pdf,[> Retouradres Postbus 20001 2500 EA Den Haag\...
1,Pdfs/nl.mnre1045.2i.2020.3.misc.1.pdf,[Datum 23 januari 2020\n\n\nBetreft Wet openba...
2,Pdfs/nl.mnre1045.2i.2020.76.misc.1.pdf,[Geachte\n\n\nIn uw brief van 20 november 2019...
3,Pdfs/nl.mnre1045.2i.2021.16.misc.1.pdf,[\n* Datum 25 maart 2021 Betreft Besluit inzak...
4,Pdfs/nl.mnre1045.2i.2022.32.misc.1.pdf,[\n* Datum 1 juni 2022 Betreft Besluit Woo-ver...


# Running PDFair (on single PDF)

In [4]:
path = "woobesluit.pdf"

pdf = PDFair.Pdf(path)  # Create Pdf class
pdf.pdf2doc(max=3)  # Run deepdoctection on first 3 pages (if available)
for i, _ in enumerate(pdf.pages):
    pdf.pages[0].detect_header()  # Detect headers for page
    pdf.pages[0].doc2md(skip_headers=True)  # Generate markdown (exclude headers)

print(pdf.pages[0].md)


[0715 16:41.50 @doctectionpipe.py:84]  INF  Processing woobesluit_0.pdf
/home/marty/miniconda3/envs/test/lib/python3.9/site-packages/torch/nn/modules/module.py:1527: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3526.)
  return forward_call(*args, **kwargs)
[0715 16:41.58 @context.py:126]  INF  ImageLayoutService total: 6.0418 sec.
[0715 16:41.58 @context.py:126]  INF  SubImageLayoutService total: 0.0001 sec.
[0715 16:41.58 @context.py:126]  INF  SubImageLayoutService total: 0.0001 sec.
[0715 16:41.58 @context.py:126]  INF  TableSegmentationService total: 0.0001 sec.
[0715 16:41.58 @context.py:126]  INF  TableSegmentationRefinementService total: 0.0001 sec.
[0715 16:41.58 @context.py:126]  INF  TextExtractionService total: 0.3993 sec.
[0715 16:41.58 @context.py:126]  INF  TextExtractionService total: 0.0007 sec.
[0715 16:41.58 @context.py:126]  INF  MatchingService

## Geachte

,


Per e-mail van 27 juli 2020 heeft u op grond van de Wet openbaarheid van bestuur (hierna: Wob) verzocht om alle bij de Rijksdienst voor het Cultureel Erfgoed (hierna: de RCE) en de Inspectie Overheidsinformatie en Erfgoed (hierna: de inspectie) aanwezige documenten over:



* Onderzoeken naar de bel, periscope en ladder van de HMS E3 in de periode 1 september 2016 tot aan heden; 
* De aan het Verenigd Koninkrijk teruggegeven artefacten van de Britse onderzeeërs HMS E3, E5 en E26 in de periode 1 september 2016 tot aan heden; 
* De teruggegeven kanonnen van de HMS Victory en La Marquise de Tourny in de periode 1 januari 2010 tot aan heden; 
* De inbeslagname van artefacten van de SMS Mainz in de periode 1 januari 2018 tot aan heden.

De ontvangst van uw verzoek is schriftelijk bevestigd bij brief van 10 augustus 2020 met kenmerk 158983. Deze brief heeft u per e-mail ontvangen. In deze brief is de beslistermijn tevens met vier weken verdaagd.


In de brief van 14 september

## Generating evaluation visualizations

In [3]:
n = 4

def generate_ngrams(eval):
    eval.pdf2txt()  
    eval.ddt_ngrams(n=n)  # Generate ngrams based on deepdoctection output
    eval.ptt_ngrams(n=n)  # Generate ngrams based on Pdf2text output (run pdf2text() first)

eval = eval.PageEval(pdf.pages[0])  # Create Evaluation class of page
generate_ngrams(eval)
eval.compare_ngrams()  # Mark tokens in missing ngrams as missing (pdf2text ngram is not in deepdoctection output)
eval.ptt.ptttoken_border(n)  # Mark every token with whether it's positioned on a border of two paragraphs

eval.visualize = types.MethodType(visualize.visualize, eval)  # Initialize standard visualization function (own functions can be used as well)
eval.visualize(ignore_header=True, ignore_pdf=False)  # Generate html visualization of page